# Heart Disease Prediction — MLOps Training Pipeline

**Task 2 — PyCaret ML Pipeline with MLflow Experiment Tracking**

This notebook trains a binary classification model to predict heart disease.
It follows a structured MLOps workflow:

| Step | Action |
|------|--------|
| 1 | Load configuration (Hydra-compatible YAML via OmegaConf) |
| 2 | Initialise MLflow tracking |
| 3 | Load data and carve out a stratified holdout set |
| 4 | Build PyCaret preprocessing pipeline (`setup`) |
| 5 | Compare models with k-fold cross-validation |
| 6 | Hyperparameter tuning on the best model |
| 7 | Evaluate and plot the tuned model **before** finalisation |
| 8 | Finalise model (retrain on full training pool) |
| 9 | Predict on truly unseen holdout data |
| 10 | Save the complete PyCaret pipeline |
| 11 | Register the model in the MLflow Model Registry |


## Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import warnings
import logging
from datetime import datetime

from omegaconf import OmegaConf           # load YAML config without @hydra.main

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

import matplotlib.pyplot as plt

from pycaret.classification import (
    setup, compare_models, tune_model,
    finalize_model, evaluate_model, predict_model,
    save_model, plot_model, pull, get_config
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix as sk_confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
logger.info('Imports complete.')

## Step 1 — Configuration

Configuration is externalised to `config_heart.yaml` and loaded with `OmegaConf`.
This avoids the `@hydra.main` decorator which is incompatible with Jupyter:
it wraps execution in a subprocess and prevents interactive cell-by-cell use.

All hyperparameters, paths, and MLflow settings are driven from the YAML,
making the notebook fully reproducible by changing a single file.

In [2]:
# OmegaConf loads the YAML into a structured, dot-accessible DictConfig
cfg = OmegaConf.load('../configs/config_heart.yaml')

logger.info('Configuration loaded:')
print(OmegaConf.to_yaml(cfg))

2026-02-22 12:32:13,417 - INFO - Configuration loaded:


data:
  raw_path: ../data/raw/heart.csv
  processed_path: data/processed/heart_processed.csv
  test_size: 0.2
  holdout_size: 0.1
  random_state: 42
model:
  experiment_name: heart_disease_experiment
  target_column: HeartDisease
  metric: AUC
  fold: 5
  n_select: 3
preprocessing:
  normalize: true
  normalize_method: zscore
  feature_selection: true
  feature_selection_method: univariate
  remove_multicollinearity: true
  multicollinearity_threshold: 0.85
  bin_numeric_features:
  - Age
  - Cholesterol
  - RestingBP
  - MaxHR
  create_interaction_features: true
tuning:
  optimize: AUC
  n_iter: 30
  random_state: 42
mlflow:
  tracking_uri: ./mlruns
  model_name: heart_disease_model
  model_stage: staging
paths:
  model_save_path: models/heart_disease_model
  reports_path: reports/figures



## Step 2 — MLflow Initialisation

MLflow is configured before any training begins so that every subsequent
step writes to the correct experiment.

> **Note:** PyCaret 3.3.x's `log_experiment=True` accesses
> `mlflow._active_run_stack.copy()` which was removed in MLflow 2.14+.
> All experiment logging is therefore done **manually** in Steps 5b and 11,
> giving identical traceability with full control over what is recorded.

In [3]:
mlflow.set_tracking_uri(cfg.mlflow.tracking_uri)
mlflow.set_experiment(cfg.model.experiment_name)

logger.info(f'MLflow tracking URI : {cfg.mlflow.tracking_uri}')
logger.info(f'MLflow experiment   : {cfg.model.experiment_name}')
logger.info('Run `mlflow ui` in the project root to inspect runs interactively.')

2026-02-22 12:32:14,510 - INFO - MLflow tracking URI : ./mlruns
2026-02-22 12:32:14,512 - INFO - MLflow experiment   : heart_disease_experiment
2026-02-22 12:32:14,512 - INFO - Run `mlflow ui` in the project root to inspect runs interactively.


## Step 3 — Data Loading and Holdout Split

A **stratified holdout set (10%)** is carved out *before* `setup()` is called.
This set is never seen by PyCaret during training, comparison, tuning, or
finalisation. It is used exclusively in Step 9 to demonstrate inference.

The remaining 90% is passed to PyCaret, which further splits it into
an internal train set (80%) and test set (20%) for cross-validation.

```
Full dataset (100%)
├── Holdout          10%  ← truly unseen; touched only in Step 9
└── Training pool    90%
    ├── CV train  ~72%  ← used during compare / tune / evaluate
    └── CV test   ~18%  ← PyCaret’s internal validation split
```

In [ ]:
data_path = Path(cfg.data.raw_path)
if not data_path.exists():
    raise FileNotFoundError(f'Data not found: {data_path}')

df = pd.read_csv(data_path)

# Drop administrative columns with no predictive value
for col in ['index', 'Patient Id']:
    if col in df.columns:
        df = df.drop(col, axis=1)

# Normalise column names (spaces -> underscores)
df.columns = df.columns.str.replace(' ', '_')

# ---------- Data-quality fix: Cholesterol = 0 ----------
# In the UCI heart disease dataset, Cholesterol=0 represents missing values,
# not actual zero cholesterol readings (clinically impossible).
# Replace with the median of non-zero values to avoid corrupting the feature
# distribution and downstream binning.
zero_chol_count = (df['Cholesterol'] == 0).sum()
if zero_chol_count > 0:
    median_chol = df.loc[df['Cholesterol'] > 0, 'Cholesterol'].median()
    df['Cholesterol'] = df['Cholesterol'].replace(0, median_chol)
    logger.info(
        f'Cholesterol: replaced {zero_chol_count} zero values with '
        f'median of non-zero values ({median_chol})'
    )

# ---------- Data-quality fix: RestingBP = 0 ----------
# RestingBP=0 is likewise clinically impossible (patient would be dead).
zero_bp_count = (df['RestingBP'] == 0).sum()
if zero_bp_count > 0:
    median_bp = df.loc[df['RestingBP'] > 0, 'RestingBP'].median()
    df['RestingBP'] = df['RestingBP'].replace(0, median_bp)
    logger.info(
        f'RestingBP: replaced {zero_bp_count} zero values with '
        f'median of non-zero values ({median_bp})'
    )

logger.info(f'Raw data shape : {df.shape}')
logger.info(f'Target column  : {cfg.model.target_column}')
logger.info(
    f'Class balance:\n'
    f'{df[cfg.model.target_column].value_counts(normalize=True).round(3)}'
)
df.head()

In [5]:
# Quick data-quality check before any processing
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Descriptive Statistics ===')
df.describe().round(2)

=== Data Types ===
Age                 int64
Sex                object
ChestPainType      object
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG         object
MaxHR               int64
ExerciseAngina     object
Oldpeak           float64
ST_Slope           object
HeartDisease        int64
dtype: object

=== Missing Values ===
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

=== Descriptive Statistics ===


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.00,918.00,918.00,918.00,918.00,918.00,918.00
mean,53.51,132.40,198.80,0.23,136.81,0.89,0.55
std,9.43,18.51,109.38,0.42,25.46,1.07,0.50
min,28.00,0.00,0.00,0.00,60.00,-2.60,0.00
25%,47.00,120.00,173.25,0.00,120.00,0.00,0.00
50%,54.00,130.00,223.00,0.00,138.00,0.60,1.00
75%,60.00,140.00,267.00,0.00,156.00,1.50,1.00
max,77.00,200.00,603.00,1.00,202.00,6.20,1.00


In [6]:
# Stratified split ensures the holdout mirrors the full dataset's class ratio
df_train_pool, df_holdout = train_test_split(
    df,
    test_size=cfg.data.holdout_size,
    random_state=cfg.data.random_state,
    stratify=df[cfg.model.target_column]
)

logger.info(f'Training pool : {df_train_pool.shape[0]} rows')
logger.info(f'Holdout set   : {df_holdout.shape[0]} rows  (excluded from all training)')
logger.info(
    f'Holdout class balance:\n'
    f'{df_holdout[cfg.model.target_column].value_counts(normalize=True).round(3)}'
)

# Persist holdout CSV for downstream auditability
holdout_raw_path = Path(cfg.data.processed_path).parent / 'heart_holdout.csv'
holdout_raw_path.parent.mkdir(parents=True, exist_ok=True)
df_holdout.to_csv(holdout_raw_path, index=False)
logger.info(f'Holdout CSV saved: {holdout_raw_path}')

2026-02-22 12:32:18,800 - INFO - Training pool : 826 rows
2026-02-22 12:32:18,801 - INFO - Holdout set   : 92 rows  (excluded from all training)
2026-02-22 12:32:18,803 - INFO - Holdout class balance:
HeartDisease
1    0.554
0    0.446
Name: proportion, dtype: float64
2026-02-22 12:32:18,817 - INFO - Holdout CSV saved: data\processed\heart_holdout.csv


## Step 4 — PyCaret Environment Setup (Preprocessing Pipeline)

`setup()` builds a **transformation pipeline** stored internally by PyCaret
and applied automatically to every dataset that passes through it.

| Technique | Config key | Rationale |
|-----------|------------|-----------|
| Z-score normalisation | `normalize=True` | Tree models are scale-invariant; LR/SVM benefit significantly |
| Univariate feature selection | `feature_selection=True` | Removes statistically weak predictors |
| Multicollinearity removal | `threshold=0.85` | 0.85 removes genuinely redundant features; 0.95 was too permissive |
| Binning: Age, Cholesterol, RestingBP, MaxHR | `bin_numeric_features` | Aligns with clinical thresholds; reduces outlier sensitivity |
| Interaction features | `feature_interaction=True` | Captures non-linear cross-feature effects |

> **** — PyCaret 3.3.x uses 
> which was removed in MLflow 2.14+. All experiment logging is done manually
> in Step 11, which gives identical traceability with full control.

In [10]:
s = setup(
    data=df_train_pool,
    target=cfg.model.target_column,
    train_size=1 - cfg.data.test_size,

    # --- Scaling and normalisation ---
    normalize=cfg.preprocessing.normalize,
    normalize_method=cfg.preprocessing.normalize_method,

    # --- Feature engineering ---
    feature_selection=cfg.preprocessing.feature_selection,
    feature_selection_method=cfg.preprocessing.feature_selection_method,
    polynomial_features=cfg.preprocessing.create_interaction_features,

    # --- Redundancy removal ---
    remove_multicollinearity=cfg.preprocessing.remove_multicollinearity,
    multicollinearity_threshold=cfg.preprocessing.multicollinearity_threshold,

    # --- Binning continuous features into clinically meaningful intervals ---
    bin_numeric_features=(
        list(cfg.preprocessing.bin_numeric_features)
        if cfg.preprocessing.bin_numeric_features else None
    ),

    # --- MLflow integration ---
    # log_experiment=False: PyCaret 3.3.x accesses mlflow._active_run_stack.copy()
    # which was removed in MLflow 2.14+. All logging is handled manually in Step 11.
    log_experiment=False,

    # --- Reproducibility ---
    session_id=cfg.data.random_state,
    verbose=True
)

logger.info('PyCaret environment initialised — transformation pipeline created.')

,Description,Value
0,Session id,42
1,Target,HeartDisease
2,Target type,Binary
3,Original data shape,"(826, 12)"
4,Transformed data shape,"(826, 3)"
5,Transformed train set shape,"(660, 3)"
6,Transformed test set shape,"(166, 3)"
7,Numeric features,6
8,Categorical features,5
9,Preprocess,True


AttributeError: 'ThreadLocalVariable' object has no attribute 'copy'

## Step 5 — Model Comparison with k-Fold Cross-Validation

Seven candidate algorithms are evaluated simultaneously using **stratified k-fold CV**.

Primary sort metric is **AUC** (changed from Accuracy):
- AUC is threshold-agnostic — measures overall discriminative power
- Robust to class imbalance
- Critical for medical use: a false negative (missed disease) costs more than a false positive

Each model's CV metrics are logged as individual MLflow runs in Step 5b.

In [ ]:
logger.info(f'Comparing models | metric={cfg.model.metric} | folds={cfg.model.fold}')

best_models = compare_models(
    include=['lightgbm', 'rf', 'xgboost', 'lr', 'nb', 'dt', 'svm'],
    sort=cfg.model.metric,
    n_select=cfg.model.n_select,
    fold=cfg.model.fold,
    cross_validation=True,
    verbose=True
)

# Normalise return value — PyCaret returns a list only when n_select > 1
if not isinstance(best_models, list):
    best_models = [best_models]

best_model = best_models[0]    # highest-AUC model used for tuning

comparison_df = pull()
logger.info(f'Best model       : {type(best_model).__name__}')
logger.info(f'Top {len(best_models)} candidates: {[type(m).__name__ for m in best_models]}')
comparison_df

### Step 5b — Log Each Compared Model to MLflow

Since PyCaret's built-in `log_experiment=True` is incompatible with MLflow 2.14+
(accesses removed `mlflow._active_run_stack`), we log each compared model
manually. This creates **one MLflow run per algorithm**, giving full visibility
in the MLflow UI — each with its cross-validation metrics.

In [ ]:
# Log each compared model as a separate MLflow run for experiment tracking
if mlflow.active_run():
    mlflow.end_run()

metric_cols = [c for c in comparison_df.columns if c not in ('Model', 'TT (Sec)')]

for idx, row in comparison_df.iterrows():
    model_name = row['Model']
    with mlflow.start_run(run_name=f'compare_{model_name}'):
        mlflow.log_param('algorithm', model_name)
        mlflow.log_param('cv_folds', cfg.model.fold)
        mlflow.log_param('sort_metric', cfg.model.metric)
        mlflow.log_param('stage', 'model_comparison')
        for col in metric_cols:
            try:
                mlflow.log_metric(f'cv_{col.lower()}', float(row[col]))
            except (ValueError, TypeError):
                pass
    logger.info(f'  Logged comparison run: {model_name}')

logger.info(f'{len(comparison_df)} model comparison runs logged to MLflow.')

## Step 6 — Hyperparameter Tuning

Random search over the best model's hyperparameter space.

- **`n_iter=30`** — increased from 10; provides adequate budget for LightGBM/XGBoost's large space
- **`optimize=AUC`** — consistent with the comparison metric
- **`choose_better=True`** — critical safeguard: PyCaret compares the tuned
  candidate against the *original* (pre-tuning) model and only accepts the
  tuned version if it actually improved the optimisation metric. This prevents
  model collapse from degenerate hyperparameter combinations.
- **`early_stopping='asha'`** — terminates unpromising trials early, reducing
  the chance of converging on a degenerate configuration

In [ ]:
logger.info(
    f'Tuning {type(best_model).__name__} | '
    f'metric={cfg.tuning.optimize} | n_iter={cfg.tuning.n_iter}'
)

tuned_model = tune_model(
    best_model,
    optimize=cfg.tuning.optimize,
    n_iter=cfg.tuning.n_iter,
    fold=cfg.model.fold,
    choose_better=True,      # Only accept tuned model if it actually improves
    early_stopping='asha',   # Terminate unpromising trials early
    verbose=True
)

tuning_df = pull()

# --- Post-tuning sanity check ---
# Detect if tuning degraded the model (kappa=0 or MCC=0 → majority-class collapse)
tuned_mean = (
    tuning_df.loc['Mean'] if 'Mean' in tuning_df.index
    else tuning_df.iloc[-2]
)
tuned_kappa = float(tuned_mean.get('Kappa', tuned_mean.get('kappa', 1)))
tuned_mcc   = float(tuned_mean.get('MCC', tuned_mean.get('mcc', 1)))

if tuned_kappa == 0 or tuned_mcc == 0:
    logger.warning(
        'TUNING DEGRADED THE MODEL (Kappa=0 or MCC=0 → majority-class collapse). '
        'Falling back to the pre-tuning best model.'
    )
    tuned_model = best_model
    tuning_df = pull()

logger.info('Hyperparameter tuning complete.')
tuning_df

## Step 7 — Model Evaluation (Before Finalisation)

**Evaluation must target `tuned_model`, not `final_model`.**

`finalize_model()` retrains on 100% of the training pool (including PyCaret's
internal test split), so there is no holdout left for evaluation after that call.
Evaluating `tuned_model` here uses PyCaret's internal CV test split — a clean holdout.

`evaluate_model()` opens an **interactive widget** in JupyterLab with tabs for
confusion matrix, AUC, classification report, learning curve, and more.

Static plots are also saved to disk so they can be uploaded as MLflow artifacts in Step 11.

In [ ]:
# Interactive evaluation widget — evaluates on PyCaret's internal CV test split
# NOTE: This is NOT the holdout set. The holdout is used only in Step 9.
evaluate_model(tuned_model)

In [ ]:
# Save static plots to disk for MLflow artifact logging
plots_dir = Path(cfg.paths.reports_path)
plots_dir.mkdir(parents=True, exist_ok=True)

_orig_dir = os.getcwd()
try:
    os.chdir(str(plots_dir))
    for plot_name in ['confusion_matrix', 'auc', 'pr', 'class_report', 'feature', 'learning']:
        logger.info(f'  Saving plot: {plot_name}')
        plot_model(tuned_model, plot=plot_name, save=True)
    logger.info(f'All plots saved to: {plots_dir}')
finally:
    os.chdir(_orig_dir)

## Step 8 — Model Finalisation

`finalize_model()` retrains the tuned estimator on the **entire training pool**
(CV train + CV test combined), maximising the data available for the production
model. The holdout set remains completely untouched.

In [ ]:
final_model = finalize_model(tuned_model)

logger.info(f'Final model type : {type(final_model).__name__}')
logger.info('Model finalised — trained on 100% of df_train_pool.')
logger.info('The holdout set (df_holdout) has NOT been seen by this model.')

## Step 9 — Predictions on Truly Unseen Data

`predict_model` is called on **`df_holdout`** — the 10% stratified split
separated in Step 3, before `setup()` was ever called.

This is the only correct demonstration of inference on unseen data.

> **Why not use `get_config('X_test')`?**  
> After `finalize_model()`, the model was retrained on both the CV-train and
> CV-test splits. Calling `predict_model` on the internal test split after
> finalisation means evaluating on data the model has already been trained on
> — producing inflated, misleading metrics.

In [ ]:
holdout_preds = predict_model(final_model, data=df_holdout)

logger.info('Predictions generated on holdout set.')
holdout_preds[[cfg.model.target_column, 'prediction_label', 'prediction_score']].head(10)

In [ ]:
y_true  = df_holdout[cfg.model.target_column]
y_pred  = holdout_preds['prediction_label']

# Sanity check — detect majority-class collapse before computing metrics
pred_counts = y_pred.value_counts()
logger.info(f'Holdout prediction distribution:\n{pred_counts.to_string()}')
if len(pred_counts) == 1:
    logger.warning(
        f'MODEL COLLAPSE DETECTED: all {len(y_pred)} holdout predictions are '
        f'class {pred_counts.index[0]}. Threshold or finalization issue — '
        f'inspect final_model decision boundary before deploying.'
    )

# Fix: PyCaret prediction_score = P(predicted_class), NOT always P(class=1).
# For predicted=1: score is already P(class=1)   → use as-is.
# For predicted=0: score = P(class=0)            → P(class=1) = 1 - score.
y_score = holdout_preds.apply(
    lambda r: r['prediction_score'] if r['prediction_label'] == 1
              else 1 - r['prediction_score'],
    axis=1
)

# --- Binary metrics (positive class = 1, i.e., heart disease) ---
holdout_metrics = {
    'holdout_accuracy' : accuracy_score(y_true, y_pred),
    'holdout_precision': precision_score(y_true, y_pred, zero_division=0),
    'holdout_recall'   : recall_score(y_true, y_pred, zero_division=0),
    'holdout_f1'       : f1_score(y_true, y_pred, zero_division=0),
    'holdout_auc'      : roc_auc_score(y_true, y_score),
}

logger.info('Holdout metrics (truly unseen data, binary — positive class = Disease):')
for k, v in holdout_metrics.items():
    logger.info(f'  {k}: {v:.4f}')

# --- Per-class classification report for full transparency ---
print('\n=== Per-Class Classification Report (Holdout) ===')
print(classification_report(
    y_true, y_pred,
    target_names=['No Disease (0)', 'Disease (1)'],
    zero_division=0
))

pd.DataFrame([holdout_metrics]).round(4)

## Step 10 — Save the Entire Pipeline

`save_model()` serialises the **complete PyCaret pipeline** — all preprocessing
transformers (normaliser, binner, feature selector, interaction generator)
plus the trained estimator — into a single `.pkl` file.

Loading this file with `load_model()` restores the full preprocessing chain,
so raw data can be passed directly for inference without any manual preprocessing.

In [ ]:
model_save_path = Path(cfg.paths.model_save_path)
model_save_path.parent.mkdir(parents=True, exist_ok=True)

save_model(final_model, str(model_save_path))
pipeline_pkl = str(model_save_path) + '.pkl'

logger.info(f'Full PyCaret pipeline saved : {pipeline_pkl}')
logger.info('Contents: preprocessing transformers + trained estimator')

# Save holdout predictions for downstream auditing / drift monitoring
holdout_pred_path = Path(cfg.data.processed_path).parent / 'heart_holdout_predictions.csv'
holdout_pred_path.parent.mkdir(parents=True, exist_ok=True)
holdout_preds.to_csv(holdout_pred_path, index=False)
logger.info(f'Holdout predictions saved   : {holdout_pred_path}')

## Step 11 — MLflow Experiment Logging and Model Registration

A dedicated **registration run** is created here, separate from the
per-model comparison runs that PyCaret auto-logged in Steps 5–6.

This run records:
- All pipeline configuration parameters
- Holdout metrics (the only unbiased estimates post-finalisation)
- The full PyCaret pipeline `.pkl` as a named artifact
- Evaluation plots as artifacts
- The model registered in the MLflow Model Registry

> **Why log both `.pkl` and `mlflow.sklearn.log_model`?**  
> `mlflow.sklearn.log_model(final_model)` logs the sklearn-compatible
> estimator for Model Registry integration. The `.pkl` from `save_model()`
> contains the **complete PyCaret pipeline** (all preprocessing steps +
> estimator) and is the artifact used for production inference.

In [ ]:
# Ensure no PyCaret-managed run is still active before opening ours
if mlflow.active_run():
    mlflow.end_run()

# Resolve actual estimator name — finalize_model wraps everything in a sklearn Pipeline,
# so type(final_model).__name__ returns "Pipeline" instead of the actual algorithm.
_last_step = final_model.steps[-1][1] if hasattr(final_model, 'steps') else final_model
model_type_name = type(_last_step).__name__
logger.info(f'Actual estimator: {model_type_name}')

run_name = f'final_registration_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

with mlflow.start_run(run_name=run_name) as run:
    run_id = run.info.run_id
    logger.info(f'MLflow registration run: {run_id}')

    # ---------------------------------------------------------- #
    # Parameters — full audit trail of every pipeline decision    #
    # ---------------------------------------------------------- #
    mlflow.log_params({
        'model_type'                 : model_type_name,
        'target_column'              : cfg.model.target_column,
        'holdout_size'               : cfg.data.holdout_size,
        'pycaret_test_size'          : cfg.data.test_size,
        'random_state'               : cfg.data.random_state,
        'cv_folds'                   : cfg.model.fold,
        'sort_metric'                : cfg.model.metric,
        'tuning_metric'              : cfg.tuning.optimize,
        'tuning_n_iter'              : cfg.tuning.n_iter,
        'normalize'                  : cfg.preprocessing.normalize,
        'normalize_method'           : cfg.preprocessing.normalize_method,
        'feature_selection'          : cfg.preprocessing.feature_selection,
        'feature_selection_method'   : cfg.preprocessing.feature_selection_method,
        'remove_multicollinearity'   : cfg.preprocessing.remove_multicollinearity,
        'multicollinearity_threshold': cfg.preprocessing.multicollinearity_threshold,
        'bin_numeric_features'       : str(list(cfg.preprocessing.bin_numeric_features)),
        'create_interaction_features': cfg.preprocessing.create_interaction_features,
        'training_pool_rows'         : len(df_train_pool),
        'holdout_rows'               : len(df_holdout),
    })

    # ---------------------------------------------------------- #
    # Metrics — holdout set only (unbiased post-finalisation)     #
    # ---------------------------------------------------------- #
    mlflow.log_metrics(holdout_metrics)
    logger.info('Holdout metrics logged.')

    # Log CV metrics from compare_models (pre-tuning, best candidate)
    try:
        if comparison_df is not None and not comparison_df.empty:
            best_row = comparison_df.iloc[0]
            for col in comparison_df.columns:
                if col not in ("Model", "TT (Sec)"):
                    try:
                        mlflow.log_metric('cv_' + col.lower().replace(' ', '_'), float(best_row[col]))
                    except Exception:
                        pass
        logger.info('Pre-tuning CV comparison metrics logged.')
    except Exception as e:
        logger.warning(f'Could not log CV comparison metrics: {e}')

    # Log CV metrics from tune_model (post-tuning — the model actually registered)
    # These are distinct from cv_ metrics above, which are pre-tuning.
    try:
        if tuning_df is not None and not tuning_df.empty:
            tuned_mean = (
                tuning_df.loc['Mean'] if 'Mean' in tuning_df.index
                else tuning_df.iloc[-2]   # second-to-last row is Mean, last is SD
            )
            for col in tuned_mean.index:
                if col not in ('Model', 'TT (Sec)'):
                    try:
                        mlflow.log_metric('tuned_cv_' + col.lower().replace(' ', '_'), float(tuned_mean[col]))
                    except Exception:
                        pass
            logger.info('Tuned model CV metrics logged.')
    except Exception as e:
        logger.warning(f'Could not log tuned CV metrics: {e}')

    # ---------------------------------------------------------- #
    # Artifact: complete PyCaret pipeline pkl                     #
    # Use pycaret.classification.load_model() to reload.          #
    # ---------------------------------------------------------- #
    mlflow.log_artifact(pipeline_pkl, artifact_path='pycaret_pipeline')
    logger.info(f'Full pipeline artifact logged: pycaret_pipeline/{Path(pipeline_pkl).name}')

    # Artifact: holdout prediction CSV (for drift monitoring / audit)
    mlflow.log_artifact(str(holdout_pred_path), artifact_path='predictions')

    # Artifact: evaluation plots from tuned_model (pre-finalization, CV test split)
    # NOTE: these plots represent tuned_model, not final_model.
    for plot_file in sorted(plots_dir.glob('*.png')):
        mlflow.log_artifact(str(plot_file), artifact_path='plots_tuned_model')
        logger.info(f'  Tuned-model plot logged: {plot_file.name}')

    # Artifact: confusion matrix from final_model on the truly unseen holdout set
    final_plots_dir = plots_dir / 'final_model'
    final_plots_dir.mkdir(exist_ok=True)
    holdout_cm_path = final_plots_dir / 'holdout_confusion_matrix.png'
    cm = sk_confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=['No Disease', 'Disease']
    ).plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix — {model_type_name} on Holdout (n={len(y_true)})')
    fig.savefig(str(holdout_cm_path), bbox_inches='tight', dpi=100)
    plt.close(fig)
    mlflow.log_artifact(str(holdout_cm_path), artifact_path='plots_final_model')
    logger.info(f'Holdout confusion matrix logged: plots_final_model/{holdout_cm_path.name}')

    # ---------------------------------------------------------- #
    # Model registration (sklearn flavor for registry support)    #
    # ---------------------------------------------------------- #
    model_info = mlflow.sklearn.log_model(
        final_model,
        artifact_path='model',
        registered_model_name=cfg.mlflow.model_name
    )
    logger.info(f'Model registered as : {cfg.mlflow.model_name}')
    logger.info(f'Model URI           : {model_info.model_uri}')
    logger.info(f'MLflow run ID       : {run_id}')

## Step 12 — Model Alias (Staging)

MLflow 2.x replaces lifecycle stages (`Staging`, `Production`) with **aliases**.
`transition_model_version_stage()` is deprecated; the replacement is
`set_registered_model_alias()` which is non-breaking and version-controlled.

In [ ]:
client = MlflowClient()

try:
    versions = client.get_latest_versions(cfg.mlflow.model_name, stages=['None'])
    if versions:
        latest_version = versions[0].version

        # MLflow 2.x — replaces deprecated transition_model_version_stage()
        client.set_registered_model_alias(
            name=cfg.mlflow.model_name,
            alias=cfg.mlflow.model_stage,    # e.g., 'staging'
            version=latest_version
        )
        logger.info(
            f"Model '{cfg.mlflow.model_name}' v{latest_version} "
            f"aliased as '{cfg.mlflow.model_stage}'."
        )
    else:
        logger.warning('No model versions found to alias.')

except Exception as e:
    logger.warning(f'Could not set alias (requires MLflow >= 2.0): {e}')

## Summary

All pipeline stages are complete. Key outputs:

In [ ]:
logger.info('=' * 65)
logger.info('HEART DISEASE PREDICTION — PIPELINE COMPLETE')
logger.info('=' * 65)
logger.info(f'  Final model       : {type(final_model).__name__}')
logger.info(f'  Holdout AUC       : {holdout_metrics["holdout_auc"]:.4f}')
logger.info(f'  Holdout F1        : {holdout_metrics["holdout_f1"]:.4f}')
logger.info(f'  Pipeline artifact : {pipeline_pkl}')
logger.info(f'  Plots             : {cfg.paths.reports_path}')
logger.info(f'  MLflow experiment : {cfg.model.experiment_name}')
logger.info(f'  Registered model  : {cfg.mlflow.model_name}')
logger.info(f'  Model alias       : {cfg.mlflow.model_stage}')
logger.info('')
logger.info(f'  MLflow UI: mlflow ui --backend-store-uri {cfg.mlflow.tracking_uri}')
logger.info('=' * 65)